In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, root_mean_squared_error
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')


c:\Users\2020d\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
train_df = pd.read_csv('trainanxietyregression.csv')
test_df = pd.read_csv('testanxietyregression.csv')


In [3]:
train_df.head()

,id,age,gender,occupation,sleep_hours,physical_activity_hrs_week,caffeine_intake_mg_day,alcohol_consumption_drinks_week,smoking,family_history_of_anxiety,stress_level,heart_rate_bpm,breathing_rate_breaths_min,sweating_level,dizziness,medication,therapy_sessions_per_month,recent_major_life_event,diet_quality,anxiety_level
0,10735,40,Female,Teacher,6.9,0.4,298,2,Yes,Yes,3,74,22,4,No,No,4,No,6,3.0
1,5937,30,Female,Lawyer,5.3,2.5,421,12,Yes,Yes,8,105,21,5,Yes,No,6,Yes,3,8.0
2,7642,31,Male,Nurse,4.9,0.5,533,6,No,Yes,10,118,25,4,Yes,No,4,Yes,1,10.0
3,3328,60,Male,Other,5.8,0.2,217,18,Yes,Yes,7,116,23,2,No,No,2,No,3,5.0
4,8681,39,Other,Musician,4.8,2.2,109,1,Yes,Yes,9,81,24,5,Yes,Yes,2,Yes,2,5.0


In [4]:
numerical_col = ["age", "sleep_hours", "physical_activity_hrs_week", "caffeine_intake_mg_day", "alcohol_consumption_drinks_week", "stress_level", "heart_rate_bpm", "breathing_rate_breaths_min", "sweating_level", "therapy_sessions_per_month", "diet_quality"]
corr = ["age", "sleep_hours", "physical_activity_hrs_week", "caffeine_intake_mg_day", "alcohol_consumption_drinks_week", "stress_level", "heart_rate_bpm", "breathing_rate_breaths_min", "sweating_level", "therapy_sessions_per_month", "diet_quality","anxiety_level"]
train_df = train_df.replace({"Yes":1, "No":0})

In [5]:
onehot = ["gender", "occupation"]
binfeature = ["smoking", "family_history_of_anxiety", "dizziness", "medication", "recent_major_life_event"]

In [6]:
x_train, x_val, y_train, y_val = train_test_split(train_df.drop(["id", "anxiety_level"], axis=1), train_df["anxiety_level"], stratify=train_df["anxiety_level"], test_size=0.2)

In [7]:
scaler = StandardScaler()
hotencoder = OneHotEncoder(sparse_output=False)

In [8]:
hotfeature = hotencoder.fit_transform(x_train[onehot])

In [9]:
scaledfeature = scaler.fit_transform(x_train[numerical_col])

In [10]:

x_out = np.hstack([x_train[binfeature].values, hotfeature, scaledfeature])

In [11]:
valhot = hotencoder.transform(x_val[onehot])
valscale = scaler.transform(x_val[numerical_col])
val_out = np.hstack([x_val[binfeature].values, valhot, valscale])

In [12]:
test_df = test_df.replace({"Yes":1, "No":0})

In [13]:
testhot = hotencoder.transform(test_df[onehot])
testscale = scaler.transform(test_df[numerical_col])
test_out = np.hstack([test_df[binfeature].values, testhot, testscale])

In [14]:
linear = LinearRegression()
linear.fit(x_out, y_train)
root_mean_squared_error(linear.predict(val_out), y_val)

np.float64(1.1151984766326868)

In [15]:
sv = SVR()
sv.fit(x_out, y_train)
root_mean_squared_error(sv.predict(val_out), y_val)

np.float64(1.0607992825382948)

In [16]:
hist = HistGradientBoostingRegressor()
hist.fit(x_out, y_train)
root_mean_squared_error(hist.predict(val_out), y_val)

np.float64(1.032063250935808)

In [17]:
mlp = MLPRegressor(hidden_layer_sizes=(20,20), activation="relu", alpha=0.01)
mlp.fit(x_out, y_train)
root_mean_squared_error(mlp.predict(val_out), y_val)

np.float64(1.0861479523183966)

In [18]:
preds_test = hist.predict(test_out)

submission = pd.DataFrame({
    'id'   : test_df['id'],
    'anxiety_level': preds_test
})

submission.to_csv('anxiety1.csv', index=False)
print('submission.csv tersimpan!')
submission.head(10)

submission.csv tersimpan!


,id,anxiety_level
0,107,5.523131
1,5484,5.054610
2,6998,9.108922
3,3984,1.573925
4,3111,2.532135
5,4040,1.553258
6,3013,5.174506
7,6607,3.398538
8,4219,2.861071
9,8750,4.182448
